In [5]:
"""
Create Kp_3h_ahead target + Time-Based Train-Test Split
This script prevents temporal leakage (scientifically correct).
"""

import pandas as pd
import os

# ---------------- CONFIG ----------------
RAW_DATA_PATH = "data/OMNI_dataset_feature_engineered_20252026.csv"
OUTPUT_DIR = "data"
TRAIN_RATIO = 0.8
# --------------------------------------


# ================== LOAD DATA ==================
df = pd.read_csv(RAW_DATA_PATH)

# Create Timestamp column
df["Timestamp"] = pd.to_datetime(df["Year"], format="%Y") \
                + pd.to_timedelta(df["DOY"] - 1, unit="D") \
                + pd.to_timedelta(df["Hour"], unit="H")

# Sort chronologically (VERY IMPORTANT)
df = df.sort_values("Timestamp").reset_index(drop=True)

print("Dataset loaded:", df.shape)
print("Time range:", df["Timestamp"].min(), "→", df["Timestamp"].max())


# ================== CREATE FUTURE TARGET ==================
# Predict Kp 3 hours ahead
df["Kp_3h_ahead"] = df["Kp"].shift(-3)

# Remove last 3 rows (no future target)
df = df.dropna(subset=["Kp_3h_ahead"]).reset_index(drop=True)

print("Target created. New dataset size:", df.shape)


# ================== SAVE FULL DATASET ==================
os.makedirs(OUTPUT_DIR, exist_ok=True)
df.to_csv(f"{OUTPUT_DIR}/full_dataset_with_target.csv", index=False)
print("Full dataset saved → data/full_dataset_with_target.csv")


# ================== TIME-BASED SPLIT ==================
split_index = int(len(df) * TRAIN_RATIO)
train_df = df.iloc[:split_index]
test_df  = df.iloc[split_index:]

print("\n--- Time-Based Split ---")
print("Train samples:", len(train_df))
print("Test samples :", len(test_df))
print("Train period:", train_df["Timestamp"].min(), "→", train_df["Timestamp"].max())
print("Test period :", test_df["Timestamp"].min(), "→", test_df["Timestamp"].max())


# ================== SAVE SPLITS ==================
train_df.to_csv(f"{OUTPUT_DIR}/train.csv", index=False)
test_df.to_csv(f"{OUTPUT_DIR}/test.csv", index=False)

print("\nTrain/Test files saved in:", OUTPUT_DIR)


/var/folders/sz/6znz8j351872v17bd_n9xhg80000gn/T/ipykernel_28319/4086779613.py:22: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  + pd.to_timedelta(df["Hour"], unit="H")


Dataset loaded: (8933, 67)
Time range: 2025-01-01 06:00:00 → 2026-01-10 23:00:00
Target created. New dataset size: (8930, 68)
Full dataset saved → data/full_dataset_with_target.csv

--- Time-Based Split ---
Train samples: 7144
Test samples : 1786
Train period: 2025-01-01 06:00:00 → 2025-10-28 03:00:00
Test period : 2025-10-28 04:00:00 → 2026-01-10 20:00:00

Train/Test files saved in: data
